# ⭐ Day 75: Proximal Policy Optimization (PPO)
Modern Reinforcement Learning Made Stable & Practical

**Python & AI Learning Path — Day 75 of 369** 🚀


## 📖 Introduction

Welcome to Day 75 of your 369-day Python & AI Learning Path! Today, we dive into one of the most influential algorithms in modern reinforcement learning: **Proximal Policy Optimization (PPO)**. Introduced by OpenAI in 2017, PPO has become the go-to algorithm for a vast range of RL applications, from training game-playing agents to fine-tuning large language models via Reinforcement Learning from Human Feedback (RLHF).

PPO was designed to solve a critical problem that plagued earlier policy gradient methods: **training instability**. Methods like REINFORCE and vanilla Actor-Critic are powerful in theory, but in practice, they suffer from large, destructive policy updates that can collapse performance suddenly. PPO addresses this with a surprisingly simple yet elegant idea: **clipping the policy update** to prevent overly aggressive changes. This small modification makes training dramatically more stable and sample-efficient.

What makes PPO truly special is its **practicality**. Unlike Trust Region Policy Optimization (TRPO), which requires complex second-order optimization and conjugate gradient methods, PPO achieves similar stability with first-order methods (standard gradient descent). This makes it easy to implement, computationally efficient, and highly scalable. It is no exaggeration to say that PPO is the "workhorse" of modern RL.

In this notebook, we will explore the theoretical foundations of PPO, implement it from scratch using PyTorch, train it on classic control environments, and visualize its behavior. By the end of today, you will understand not just *how* PPO works, but *why* it works—and why it dominates the RL landscape. Let's get started! 🧠📈


## 📋 Table of Contents

1. [Limitations of Vanilla Policy Gradient and Actor-Critic Methods](#1-limitations-of-vanilla-policy-gradient-and-actor-critic-methods)
2. [The Core Idea of PPO: Clipped Surrogate Objective](#2-the-core-idea-of-ppo-clipped-surrogate-objective)
3. [PPO Objective Function Explained](#3-ppo-objective-function-explained)
4. [PPO Algorithm Steps (Actor-Critic Style)](#4-ppo-algorithm-steps-actor-critic-style)
5. [Implementing PPO (Simplified Version with PyTorch)](#5-implementing-ppo-simplified-version-with-pytorch)
6. [Training PPO on CartPole Environment](#6-training-ppo-on-cartpole-environment)
7. [Visualizing Training Progress](#7-visualizing-training-progress)
8. [Hyperparameters in PPO](#8-hyperparameters-in-ppo)
9. [Comparison with DQN and REINFORCE](#9-comparison-with-dqn-and-reinforce)
10. [Real-World Applications and Why PPO is So Popular](#10-real-world-applications-and-why-ppo-is-so-popular)
11. [🛠️ Hands-On Exercises](#hands-on-exercises)
12. [Solutions (Review After Attempting)](#solutions-review-after-attempting)


## 1. Limitations of Vanilla Policy Gradient and Actor-Critic Methods

Before we appreciate PPO, we must understand the problems it solves. Let's review the key limitations of earlier policy gradient methods:

### 1.1 Destructively Large Policy Updates
Vanilla policy gradient methods (like REINFORCE) update the policy in the direction of the gradient:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \nabla_\theta \log \pi_\theta(a|s) \cdot A(s,a) \right]$$
However, a single large gradient step can push the policy into a **bad region of parameter space** from which it never recovers. There is no mechanism to limit how far the policy moves in one update.

### 1.2 Sample Inefficiency
REINFORCE is a Monte Carlo method requiring full episodes before updating. Actor-Critic methods improve this by using bootstrapping, but they still require careful tuning of learning rates to avoid divergence.

### 1.3 Sensitivity to Hyperparameters
The learning rate in vanilla policy gradients is notoriously sensitive. Too high, and the policy collapses; too low, and learning is painfully slow. Different environments may require vastly different learning rates.

### 1.4 The Need for Trust Regions
TRPO introduced the concept of a "trust region"—limiting policy updates to ensure the new policy doesn't diverge too far from the old one. However, TRPO is complex to implement and computationally expensive due to its use of Fisher information matrices and conjugate gradients.

**PPO's breakthrough** was to achieve TRPO's stability with the simplicity of first-order methods. 💡


## 2. The Core Idea of PPO: Clipped Surrogate Objective

PPO's central innovation is replacing the vanilla policy gradient objective with a **clipped surrogate objective** that penalizes large policy changes.

### 2.1 The Probability Ratio
Define the probability ratio between the new policy and the old policy:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$$
- If $r_t(\theta) > 1$, the new policy is **more likely** to take action $a_t$ in state $s_t$.
- If $r_t(\theta) < 1$, the new policy is **less likely**.
- If $r_t(\theta) = 1$, the policy hasn't changed for this state-action pair.

### 2.2 The Unclipped Surrogate Objective
The naive approach would be to maximize:
$$L^{CPI}(\theta) = \mathbb{E}_t\left[ r_t(\theta) \cdot A_t \right]$$
where $A_t$ is the advantage estimate. This is called the **Conservative Policy Iteration** objective. Without constraints, this can lead to destructively large updates.

### 2.3 The Clipped Surrogate Objective
PPO modifies this by **clipping** the ratio to a small interval around 1:
$$L^{CLIP}(\theta) = \mathbb{E}_t\left[ \min\left( r_t(\theta) \cdot A_t, \; r_t^{clip}(\theta) \cdot A_t \right) \right]$$
where:
$$r_t^{clip}(\theta) = \text{clip}\left( r_t(\theta), 1 - \epsilon, 1 + \epsilon \right)$$

**The intuition:**
- If the advantage $A_t > 0$ (action was good), we want to increase the probability, but **not too much** (clipped at $1 + \epsilon$).
- If the advantage $A_t < 0$ (action was bad), we want to decrease the probability, but **not too much** (clipped at $1 - \epsilon$).
- The $\min$ operator ensures we take the **pessimistic** (lower) bound, preventing the objective from benefiting from overly large changes in either direction.

This simple clipping mechanism is the secret to PPO's stability! 🎯


In [ ]:
# 💡 Visualization: Understanding the Clipped Surrogate Objective
import numpy as np
import matplotlib.pyplot as plt

def clipped_objective(ratio, advantage, epsilon=0.2):
    """Compute the PPO clipped surrogate objective for a single (s,a) pair."""
    clipped_ratio = np.clip(ratio, 1 - epsilon, 1 + epsilon)
    unclipped = ratio * advantage
    clipped = clipped_ratio * advantage
    return np.minimum(unclipped, clipped)

# Create a range of probability ratios
ratios = np.linspace(0.5, 1.5, 500)
epsilon = 0.2

# Plot for positive and negative advantages
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, adv, title in zip(axes, [1.0, -1.0], ['Positive Advantage (A > 0)', 'Negative Advantage (A < 0)']):
    unclipped = ratios * adv
    clipped = np.clip(ratios, 1 - epsilon, 1 + epsilon) * adv
    objective = np.minimum(unclipped, clipped)
    
    ax.plot(ratios, unclipped, 'b--', linewidth=2, label=r'$r_t(\theta) \cdot A_t$ (unclipped)')
    ax.plot(ratios, clipped, 'g:', linewidth=2, label=r'$r_t^{clip}(\theta) \cdot A_t$ (clipped)')
    ax.plot(ratios, objective, 'r-', linewidth=3, label=r'$L^{CLIP}$ (objective)')
    ax.axvline(1 + epsilon, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1 - epsilon, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.0, color='black', linestyle='--', alpha=0.3)
    ax.fill_betweenx(ax.get_ylim(), 1 - epsilon, 1 + epsilon, alpha=0.1, color='green')
    ax.set_xlabel('Probability Ratio $r_t(\theta)$', fontsize=12)
    ax.set_ylabel('Objective Value', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.5, 1.5)

plt.suptitle('PPO Clipped Surrogate Objective Visualization', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 3. PPO Objective Function Explained

The complete PPO objective combines three components to balance policy improvement, value estimation, and exploration:

### 3.1 Full PPO Objective
$$L^{PPO}(\theta) = \mathbb{E}_t\left[ L_t^{CLIP}(\theta) \right] - c_1 \cdot L_t^{VF}(\theta) + c_2 \cdot S[\pi_\theta](s_t)$$

Where:
1. **$L_t^{CLIP}(\theta)$**: The clipped surrogate objective (policy loss)
2. **$L_t^{VF}(\theta)$**: The value function loss (MSE between predicted and target values)
3. **$S[\pi_\theta](s_t)$**: The entropy bonus (encourages exploration)
4. **$c_1, c_2$**: Coefficients controlling the relative importance

### 3.2 Value Function Loss
$$L_t^{VF}(\theta) = \left( V_\theta(s_t) - V_t^{target} \right)^2$$
The value network is trained to predict the expected return, typically using the return computed via GAE (Generalized Advantage Estimation).

### 3.3 Entropy Bonus
$$S[\pi_\theta](s_t) = -\sum_a \pi_\theta(a|s_t) \log \pi_\theta(a|s_t)$$
High entropy means the policy is uncertain/exploratory. By adding entropy to the objective, we encourage the agent to keep exploring rather than prematurely converging to a suboptimal deterministic policy.

### 3.4 Generalized Advantage Estimation (GAE)
PPO typically uses GAE to compute advantage estimates, which provides a bias-variance tradeoff:
$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}^V$$
where $\delta_t^V = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD residual.

- $\lambda = 0$: High bias, low variance (like TD(0))
- $\lambda = 1$: Low bias, high variance (like Monte Carlo)
- Typical value: $\lambda = 0.95$


In [ ]:
# 📐 Mathematical implementation of PPO components
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    Compute Generalized Advantage Estimation (GAE).
    
    Args:
        rewards: np.array of shape (T,) — rewards at each timestep
        values: np.array of shape (T+1,) — value estimates (includes bootstrap value at end)
        dones: np.array of shape (T,) — done flags (1 if episode ended, 0 otherwise)
        gamma: discount factor
        lam: GAE lambda parameter
    
    Returns:
        advantages: np.array of shape (T,)
        returns: np.array of shape (T,) — advantage + value (target for value function)
    """
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_gae = 0.0
    
    for t in reversed(range(len(rewards))):
        if dones[t]:
            next_value = 0.0
            last_gae = 0.0
        else:
            next_value = values[t + 1]
        
        delta = rewards[t] + gamma * next_value - values[t]
        advantages[t] = last_gae = delta + gamma * lam * last_gae
    
    returns = advantages + values[:-1]
    return advantages, returns

# Test GAE computation with a simple example
rewards = np.array([1.0, 0.0, 1.0, 0.0, 1.0])
values = np.array([0.5, 0.6, 0.55, 0.7, 0.65, 0.0])  # T+1 values, last is bootstrap
dones = np.array([0, 0, 0, 0, 1])  # Episode ends at last step

advantages, returns = compute_gae(rewards, values, dones, gamma=0.99, lam=0.95)
print("Rewards:", rewards)
print("Values:", values[:-1])
print("Advantages (GAE):", np.round(advantages, 4))
print("Returns (targets):", np.round(returns, 4))


## 4. PPO Algorithm Steps (Actor-Critic Style)

PPO follows an Actor-Critic architecture with a clipped objective. Here is the complete algorithm:

### Algorithm: PPO (Clip Version)

**Input:** Initial policy parameters $\theta_0$, initial value function parameters $\phi_0$
**Hyperparameters:** Clipping threshold $\epsilon$, learning rates $\alpha_\theta, \alpha_\phi$, batch size $B$, epochs $K$, timesteps per update $T$

**For** iteration $= 1, 2, ...$ **do:**
1. **Collect Trajectories:** Run policy $\pi_{\theta_{old}}$ for $T$ timesteps, collecting $(s_t, a_t, r_t, s_{t+1}, done_t)$
2. **Compute Advantages:** Use GAE to compute $\hat{A}_t$ and returns $R_t$ for all timesteps
3. **Normalize Advantages:** $\hat{A}_t \leftarrow (\hat{A}_t - \mu_A) / \sigma_A$
4. **Update Policy (for $K$ epochs):**
   - Shuffle and split data into mini-batches
   - For each mini-batch:
     - Compute probability ratio $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$
     - Compute clipped surrogate loss: $L^{CLIP} = \min(r_t \hat{A}_t, r_t^{clip} \hat{A}_t)$
     - Compute value loss: $L^{VF} = (V_\phi(s_t) - R_t)^2$
     - Compute entropy bonus: $S = \text{Entropy}(\pi_\theta(\cdot|s_t))$
     - Total loss: $L = -L^{CLIP} + c_1 L^{VF} - c_2 S$
     - Update $\theta$ and $\phi$ via gradient descent on $L$
5. **Repeat** until convergence

### Key Differences from Standard Actor-Critic:
- **Multiple epochs per batch:** Unlike standard AC which updates once per sample, PPO reuses the same batch for $K$ epochs (typically 3-10).
- **Clipping prevents over-optimization:** The clipped objective naturally limits how much the policy can change for any single state-action pair.
- **No need for trust region constraints:** The clipping is a soft constraint that is much easier to implement than TRPO's hard KL constraint.


## 5. Implementing PPO (Simplified Version with PyTorch)

Let's build a clean, educational implementation of PPO. We'll create:
1. An Actor-Critic neural network
2. A memory buffer for trajectory storage
3. The PPO update function with clipping
4. A training loop


In [ ]:
# 🏗️ Step 1: Import Libraries and Setup
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# 🧠 Step 2: Actor-Critic Network Architecture
class ActorCritic(nn.Module):
    """
    Shared-body Actor-Critic network for PPO.
    - Actor head outputs action probabilities (for discrete actions)
    - Critic head outputs state value estimate
    """
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(ActorCritic, self).__init__()
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Actor head: outputs logits for action probabilities
        self.actor = nn.Linear(hidden_dim, action_dim)
        
        # Critic head: outputs state value
        self.critic = nn.Linear(hidden_dim, 1)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
            nn.init.zeros_(module.bias)
    
    def forward(self, state):
        """Forward pass returning both action logits and state value."""
        features = self.shared(state)
        action_logits = self.actor(features)
        state_value = self.critic(features)
        return action_logits, state_value
    
    def get_action_and_value(self, state, action=None):
        """
        Get action distribution, sample action, log prob, entropy, and value.
        If action is provided, compute log prob and entropy for that action.
        """
        action_logits, state_value = self.forward(state)
        dist = torch.distributions.Categorical(logits=action_logits)
        
        if action is None:
            action = dist.sample()
        
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        
        return action, log_prob, entropy, state_value.squeeze(-1)

# Test the network
test_net = ActorCritic(state_dim=4, action_dim=2).to(device)
test_state = torch.randn(1, 4).to(device)
action, log_prob, entropy, value = test_net.get_action_and_value(test_state)
print(f"Sample action: {action.item()}, Log prob: {log_prob.item():.4f}, Entropy: {entropy.item():.4f}, Value: {value.item():.4f}")


In [ ]:
# 📦 Step 3: Trajectory Memory Buffer
class PPOMemory:
    """Memory buffer for storing trajectories collected by the policy."""
    
    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.values = []
        self.log_probs = []
        self.dones = []
    
    def add(self, state, action, reward, value, log_prob, done):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.values.append(value)
        self.log_probs.append(log_prob)
        self.dones.append(done)
    
    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.rewards.clear()
        self.values.clear()
        self.log_probs.clear()
        self.dones.clear()
    
    def get_batches(self, batch_size):
        """Generate random mini-batches for PPO updates."""
        n = len(self.states)
        indices = np.arange(n)
        np.random.shuffle(indices)
        
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_idx = indices[start:end]
            yield batch_idx
    
    def __len__(self):
        return len(self.states)


In [ ]:
# ⚙️ Step 4: PPO Agent Class
class PPOAgent:
    """
    Proximal Policy Optimization Agent.
    
    Hyperparameters:
    - lr: learning rate
    - gamma: discount factor
    - gae_lambda: GAE lambda parameter
    - clip_epsilon: PPO clipping parameter
    - entropy_coef: entropy bonus coefficient
    - value_coef: value loss coefficient
    - epochs: number of optimization epochs per batch
    - batch_size: mini-batch size for updates
    """
    
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, 
                 gae_lambda=0.95, clip_epsilon=0.2, entropy_coef=0.01,
                 value_coef=0.5, epochs=4, batch_size=64):
        
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef
        self.epochs = epochs
        self.batch_size = batch_size
        
        self.policy = ActorCritic(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr, eps=1e-5)
        
        self.memory = PPOMemory()
    
    def select_action(self, state):
        """Select action using current policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action, log_prob, _, value = self.policy.get_action_and_value(state_tensor)
        return action.item(), log_prob.item(), value.item()
    
    def compute_advantages(self):
        """Compute GAE advantages and returns from stored trajectories."""
        rewards = np.array(self.memory.rewards, dtype=np.float32)
        values = np.array(self.memory.values + [0.0], dtype=np.float32)  # Append bootstrap value
        dones = np.array(self.memory.dones, dtype=np.float32)
        
        advantages, returns = compute_gae(rewards, values, dones, self.gamma, self.gae_lambda)
        
        # Normalize advantages (important for training stability!)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return advantages, returns
    
    def update(self):
        """Perform PPO update on collected trajectories."""
        # Convert memory to tensors
        states = torch.FloatTensor(np.array(self.memory.states)).to(device)
        actions = torch.LongTensor(self.memory.actions).to(device)
        old_log_probs = torch.FloatTensor(self.memory.log_probs).to(device)
        
        advantages, returns = self.compute_advantages()
        advantages = torch.FloatTensor(advantages).to(device)
        returns = torch.FloatTensor(returns).to(device)
        
        # Store metrics for logging
        policy_losses = []
        value_losses = []
        entropy_bonuses = []
        approx_kls = []
        clip_fractions = []
        
        # PPO update loop: multiple epochs over the same data
        for epoch in range(self.epochs):
            for batch_idx in self.memory.get_batches(self.batch_size):
                batch_states = states[batch_idx]
                batch_actions = actions[batch_idx]
                batch_old_log_probs = old_log_probs[batch_idx]
                batch_advantages = advantages[batch_idx]
                batch_returns = returns[batch_idx]
                
                # Evaluate current policy on batch data
                _, log_probs, entropy, state_values = self.policy.get_action_and_value(
                    batch_states, batch_actions
                )
                
                # Compute probability ratio
                ratio = torch.exp(log_probs - batch_old_log_probs)
                
                # Clipped surrogate objective
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                
                # Value function loss
                value_loss = nn.MSELoss()(state_values, batch_returns)
                
                # Entropy bonus (encourages exploration)
                entropy_loss = -entropy.mean()
                
                # Total loss
                loss = policy_loss + self.value_coef * value_loss + self.entropy_coef * entropy_loss
                
                # Optimization step
                self.optimizer.zero_grad()
                loss.backward()
                # Gradient clipping for stability
                nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=0.5)
                self.optimizer.step()
                
                # Logging metrics
                with torch.no_grad():
                    approx_kl = ((ratio - 1) - ratio.log()).mean().item()
                    clip_fraction = ((ratio - 1.0).abs() > self.clip_epsilon).float().mean().item()
                
                policy_losses.append(policy_loss.item())
                value_losses.append(value_loss.item())
                entropy_bonuses.append(entropy.mean().item())
                approx_kls.append(approx_kl)
                clip_fractions.append(clip_fraction)
        
        # Clear memory after update
        self.memory.clear()
        
        return {
            'policy_loss': np.mean(policy_losses),
            'value_loss': np.mean(value_losses),
            'entropy': np.mean(entropy_bonuses),
            'approx_kl': np.mean(approx_kls),
            'clip_fraction': np.mean(clip_fractions)
        }


## 6. Training PPO on CartPole Environment

Now let's train our PPO agent on the classic **CartPole-v1** environment. This is a great testbed because:
- It's simple enough to train quickly
- It clearly demonstrates policy learning
- It allows us to visualize PPO's stability


In [ ]:
# 🎮 Step 5: Setup CartPole Environment
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"Environment: CartPole-v1")
print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")


In [ ]:
# 🏋️ Step 6: Training Configuration and Loop
# Hyperparameters
CONFIG = {
    'lr': 3e-4,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_epsilon': 0.2,
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    'epochs': 4,
    'batch_size': 64,
    'timesteps_per_update': 2048,  # Collect this many steps before updating
    'total_timesteps': 200_000,
    'max_episode_steps': 500
}

agent = PPOAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    lr=CONFIG['lr'],
    gamma=CONFIG['gamma'],
    gae_lambda=CONFIG['gae_lambda'],
    clip_epsilon=CONFIG['clip_epsilon'],
    entropy_coef=CONFIG['entropy_coef'],
    value_coef=CONFIG['value_coef'],
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size']
)

print("PPO Agent initialized with configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


In [ ]:
# 🚀 Step 7: Main Training Loop
# Metrics tracking
episode_rewards = []
episode_lengths = []
policy_losses = []
value_losses = []
entropies = []
clip_fractions = []
approx_kls = []
timesteps = []

global_step = 0
episode = 0
current_episode_reward = 0
current_episode_length = 0

state, _ = env.reset(seed=SEED)

print("Starting PPO training on CartPole-v1...")
print("=" * 60)

while global_step < CONFIG['total_timesteps']:
    # Collect trajectories
    for _ in range(CONFIG['timesteps_per_update']):
        action, log_prob, value = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Store transition
        agent.memory.add(state, action, reward, value, log_prob, done)
        
        state = next_state
        current_episode_reward += reward
        current_episode_length += 1
        global_step += 1
        
        if done or current_episode_length >= CONFIG['max_episode_steps']:
            episode_rewards.append(current_episode_reward)
            episode_lengths.append(current_episode_length)
            
            state, _ = env.reset()
            current_episode_reward = 0
            current_episode_length = 0
            episode += 1
    
    # Update policy after collecting enough timesteps
    metrics = agent.update()
    
    # Log metrics
    policy_losses.append(metrics['policy_loss'])
    value_losses.append(metrics['value_loss'])
    entropies.append(metrics['entropy'])
    clip_fractions.append(metrics['clip_fraction'])
    approx_kls.append(metrics['approx_kl'])
    timesteps.append(global_step)
    
    # Print progress every few updates
    if len(timesteps) % 5 == 0 or global_step >= CONFIG['total_timesteps']:
        recent_rewards = episode_rewards[-20:] if len(episode_rewards) >= 20 else episode_rewards
        mean_reward = np.mean(recent_rewards) if recent_rewards else 0
        print(f"Step {global_step:>6} | Episodes: {episode:>3} | "
              f"Mean Reward (last 20): {mean_reward:>7.2f} | "
              f"Policy Loss: {metrics['policy_loss']:>7.4f} | "
              f"Value Loss: {metrics['value_loss']:>7.4f} | "
              f"Entropy: {metrics['entropy']:>5.3f} | "
              f"Clip Frac: {metrics['clip_fraction']:>5.3f}")

print("=" * 60)
print(f"Training complete! Total episodes: {episode}")


## 7. Visualizing Training Progress

Let's create comprehensive visualizations to understand how PPO learns:
1. Episode rewards over time
2. Policy loss and value loss curves
3. Entropy decay (exploration vs exploitation)
4. Clip fraction (how often clipping is active)
5. Approximate KL divergence (policy change magnitude)


In [ ]:
# 📊 Step 8: Plot Training Metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('PPO Training Progress on CartPole-v1', fontsize=16, fontweight='bold')

# 1. Episode Rewards
ax = axes[0, 0]
ax.plot(episode_rewards, alpha=0.3, color='blue', label='Raw')
# Moving average
window = 20
if len(episode_rewards) >= window:
    ma = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(episode_rewards)), ma, color='red', linewidth=2, label=f'MA({window})')
ax.axhline(y=500, color='green', linestyle='--', alpha=0.5, label='Max Reward (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('📈 Episode Rewards')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Policy Loss
ax = axes[0, 1]
ax.plot(timesteps, policy_losses, color='purple', linewidth=2)
ax.set_xlabel('Timesteps')
ax.set_ylabel('Policy Loss')
ax.set_title('🎯 Policy Loss (Clipped Surrogate)')
ax.grid(True, alpha=0.3)

# 3. Value Loss
ax = axes[0, 2]
ax.plot(timesteps, value_losses, color='orange', linewidth=2)
ax.set_xlabel('Timesteps')
ax.set_ylabel('Value Loss (MSE)')
ax.set_title('💰 Value Function Loss')
ax.grid(True, alpha=0.3)

# 4. Entropy
ax = axes[1, 0]
ax.plot(timesteps, entropies, color='green', linewidth=2)
ax.set_xlabel('Timesteps')
ax.set_ylabel('Entropy')
ax.set_title('🎲 Policy Entropy (Exploration)')
ax.grid(True, alpha=0.3)

# 5. Clip Fraction
ax = axes[1, 1]
ax.plot(timesteps, clip_fractions, color='red', linewidth=2)
ax.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='Target ~10%')
ax.set_xlabel('Timesteps')
ax.set_ylabel('Clip Fraction')
ax.set_title('✂️ Clipping Activity')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Approximate KL Divergence
ax = axes[1, 2]
ax.plot(timesteps, approx_kls, color='brown', linewidth=2)
ax.set_xlabel('Timesteps')
ax.set_ylabel('Approx KL')
ax.set_title('📏 Policy Change (KL Divergence)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# 🎬 Step 9: Visualize the Learned Policy in Action
def evaluate_policy(agent, env_name='CartPole-v1', num_episodes=5, render=False):
","    """Evaluate the trained policy."""
    eval_env = gym.make(env_name, render_mode='human' if render else None)
    eval_rewards = []
    
    for ep in range(num_episodes):
        state, _ = eval_env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                action_logits, _ = agent.policy(state_tensor)
                action = torch.argmax(action_logits, dim=-1).item()
            
            state, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        eval_rewards.append(episode_reward)
        print(f"Evaluation Episode {ep + 1}: Reward = {episode_reward}")
    
    eval_env.close()
    print(f"\nMean Evaluation Reward: {np.mean(eval_rewards):.2f} (+/- {np.std(eval_rewards):.2f})")
    return eval_rewards

# Run evaluation
print("Evaluating trained PPO policy...")
eval_rewards = evaluate_policy(agent, num_episodes=10)


In [ ]:
# 🔍 Step 10: Analyze the Effect of Clipping
# Let's create a visualization showing how clipping affects the objective
# for different advantage values and ratios

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epsilon = CONFIG['clip_epsilon']
ratios = np.linspace(0.5, 1.5, 200)

for ax, advantage, title in zip(axes, [2.0, -2.0], ['Good Action (A = +2.0)', 'Bad Action (A = -2.0)']):
    unclipped = ratios * advantage
    clipped = np.clip(ratios, 1 - epsilon, 1 + epsilon) * advantage
    objective = np.minimum(unclipped, clipped)
    
    ax.plot(ratios, unclipped, 'b--', linewidth=2, label='Unclipped: $r_t \\cdot A_t$')
    ax.plot(ratios, clipped, 'g:', linewidth=2, label='Clipped: $r_t^{clip} \\cdot A_t$')
    ax.plot(ratios, objective, 'r-', linewidth=3, label='PPO Objective: $\\min$')
    
    ax.axvline(1 + epsilon, color='gray', linestyle='-', alpha=0.4)
    ax.axvline(1 - epsilon, color='gray', linestyle='-', alpha=0.4)
    ax.axvline(1.0, color='black', linestyle='--', alpha=0.3, label='Ratio = 1.0')
    ax.fill_betweenx(ax.get_ylim(), 1 - epsilon, 1 + epsilon, alpha=0.1, color='green', label='Trust Region')
    
    ax.set_xlabel('Probability Ratio $r_t(\\theta)$', fontsize=12)
    ax.set_ylabel('Objective Value', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.5, 1.5)

plt.suptitle('Why PPO Clipping Works: Pessimistic Bound on Policy Updates', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Hyperparameters in PPO

PPO has several critical hyperparameters that significantly affect performance. Understanding them is key to successful RL training:

### 8.1 Clipping Parameter ($\epsilon$)
- **Typical values:** 0.1 to 0.3 (default: 0.2)
- **Effect:** Controls how far the policy can deviate from the old policy in a single update.
- **Too high:** Large, potentially destructive policy updates; training instability.
- **Too low:** Overly conservative updates; slow learning, poor sample efficiency.
- **Rule of thumb:** Start with 0.2. If training is unstable, reduce to 0.1. If learning is too slow, try 0.3.

### 8.2 GAE Lambda ($\lambda$)
- **Typical values:** 0.9 to 0.99 (default: 0.95)
- **Effect:** Controls bias-variance tradeoff in advantage estimation.
- **Low ($\lambda \approx 0$):** High bias, low variance. More stable but may miss long-term dependencies.
- **High ($\lambda \approx 1$):** Low bias, high variance. Better long-term credit assignment but noisier.

### 8.3 Entropy Coefficient ($c_2$)
- **Typical values:** 0.001 to 0.1 (default: 0.01)
- **Effect:** Encourages exploration by rewarding high-entropy (uncertain) policies.
- **Too high:** Excessive exploration; policy never commits to good actions.
- **Too low:** Premature convergence to suboptimal deterministic policies.
- **Decay:** Often decayed over time as the policy improves.

### 8.4 Value Function Coefficient ($c_1$)
- **Typical values:** 0.25 to 1.0 (default: 0.5)
- **Effect:** Balances the importance of value function learning vs policy improvement.
- **Note:** Some implementations use separate optimizers for actor and critic with different learning rates instead.

### 8.5 Learning Rate
- **Typical values:** 1e-4 to 1e-3 (default: 3e-4)
- **Effect:** Standard deep learning hyperparameter. PPO is generally more robust to learning rate choices than vanilla policy gradients due to clipping.
- **Scheduling:** Learning rate annealing (decay over time) often improves final performance.

### 8.6 Batch Size and Update Epochs
- **Timesteps per update:** 2048 to 4096 (default: 2048)
- **Epochs ($K$):** 3 to 10 (default: 4)
- **Mini-batch size:** 32 to 512 (default: 64)
- **Effect:** More epochs = more optimization per sample (better sample efficiency) but risk overfitting to the current batch. Clipping helps prevent this.

### 8.7 Discount Factor ($\gamma$)
- **Typical values:** 0.99 to 0.999
- **Effect:** Standard RL discount factor. Higher values prioritize long-term rewards.


In [ ]:
# 🔬 Step 11: Hyperparameter Sensitivity Analysis
# Let's compare PPO with different clip values to see the effect on training stability

def train_ppo_with_config(clip_epsilon, total_timesteps=100_000, seed=SEED):
    """Train PPO with a specific clip epsilon and return reward history."""
    env = gym.make('CartPole-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    agent = PPOAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_epsilon=clip_epsilon,
        entropy_coef=0.01,
        value_coef=0.5,
        epochs=4,
        batch_size=64
    )
    
    episode_rewards = []
    global_step = 0
    episode = 0
    current_reward = 0
    state, _ = env.reset(seed=seed)
    
    while global_step < total_timesteps:
        for _ in range(2048):
            action, log_prob, value = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.memory.add(state, action, reward, value, log_prob, done)
            state = next_state
            current_reward += reward
            global_step += 1
            
            if done:
                episode_rewards.append(current_reward)
                current_reward = 0
                state, _ = env.reset()
                episode += 1
        
        agent.update()
    
    env.close()
    return episode_rewards

# Compare different clip values
print("Running hyperparameter comparison (this may take a few minutes)...")
clip_values = [0.1, 0.2, 0.3]
results = {}

for clip in clip_values:
    print(f"Training with clip_epsilon={clip}...")
    rewards = train_ppo_with_config(clip, total_timesteps=80_000)
    results[clip] = rewards
    print(f"  Completed: {len(rewards)} episodes, final mean reward: {np.mean(rewards[-20:]):.2f}")


In [ ]:
# Plot comparison
plt.figure(figsize=(12, 6))
colors = ['blue', 'green', 'red']
window = 20

for clip, color in zip(clip_values, colors):
    rewards = results[clip]
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(rewards)), ma, color=color, linewidth=2, 
                label=f'clip_ε = {clip}')
    plt.plot(rewards, color=color, alpha=0.2)

plt.axhline(y=500, color='black', linestyle='--', alpha=0.5, label='Max Reward')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Total Reward', fontsize=12)
plt.title('Effect of Clip Epsilon on PPO Training Stability', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Comparison with DQN and REINFORCE

Understanding where PPO fits in the RL algorithm landscape helps appreciate its design:

### 9.1 PPO vs REINFORCE
| Aspect | REINFORCE | PPO |
|--------|-----------|-----|
| **Update frequency** | Per episode (Monte Carlo) | Per fixed timesteps (mini-batch) |
| **Sample efficiency** | Very low | High (reuses samples for multiple epochs) |
| **Stability** | Poor (high variance, no clipping) | Excellent (clipped surrogate objective) |
| **Learning rate sensitivity** | Very high | Moderate (clipping provides robustness) |
| **Baseline** | Optional | Required (Actor-Critic) |
| **Implementation complexity** | Simple | Moderate |

### 9.2 PPO vs DQN
| Aspect | DQN | PPO |
|--------|-----|-----|
| **Policy type** | Off-policy (value-based) | On-policy (policy-based) |
| **Action space** | Discrete only | Discrete and continuous |
| **Sample efficiency** | High (experience replay) | Lower (on-policy, no replay) |
| **Stability** | Moderate (target networks help) | High (clipping) |
| **Exploration** | Epsilon-greedy | Stochastic policy (entropy) |
| **Continuous control** | Requires adaptations (e.g., DDPG) | Native support |
| **Hyperparameter sensitivity** | Moderate | Moderate |

### 9.3 When to Choose PPO
- **Discrete action spaces** where you want stable, reliable training
- **Continuous control** tasks (robotics, locomotion)
- **Environments with sparse rewards** (GAE helps with credit assignment)
- **When you need a robust default** that works out-of-the-box
- **RLHF and fine-tuning LLMs** (PPO is the standard algorithm)

### 9.4 When DQN Might Be Better
- **Sample efficiency is critical** and you can afford off-policy learning
- **Discrete action spaces** with dense rewards
- **When you have a large replay buffer** and can leverage past experiences


In [ ]:
# 📊 Step 12: Algorithm Comparison Visualization
# Create a radar chart comparing key properties

import numpy as np
import matplotlib.pyplot as plt

categories = ['Sample\nEfficiency', 'Training\nStability', 'Implementation\nEase', 
              'Continuous\nActions', 'Convergence\nSpeed', 'Final\nPerformance']
N = len(categories)

# Scores out of 10 (subjective but representative)
ppo_scores = [7, 9, 7, 9, 8, 8]
dqn_scores = [9, 6, 8, 3, 7, 7]
reinforce_scores = [3, 3, 9, 5, 4, 5]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ppo_scores += ppo_scores[:1]
dqn_scores += dqn_scores[:1]
reinforce_scores += reinforce_scores[:1]

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
ax.plot(angles, ppo_scores, 'o-', linewidth=2, label='PPO', color='green')
ax.fill(angles, ppo_scores, alpha=0.15, color='green')
ax.plot(angles, dqn_scores, 'o-', linewidth=2, label='DQN', color='blue')
ax.fill(angles, dqn_scores, alpha=0.15, color='blue')
ax.plot(angles, reinforce_scores, 'o-', linewidth=2, label='REINFORCE', color='red')
ax.fill(angles, reinforce_scores, alpha=0.15, color='red')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=9)
ax.grid(True)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)

plt.title('RL Algorithm Comparison: PPO vs DQN vs REINFORCE', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## 10. Real-World Applications and Why PPO is So Popular

PPO has become the dominant RL algorithm in both research and industry. Here's why:

### 10.1 Game Playing and Simulations
- **OpenAI Five:** Defeated world champions in Dota 2 using PPO
- **AlphaStar (DeepMind):** Used policy gradient variants (including PPO ideas) for StarCraft II
- **General game AI:** PPO is the default choice for training agents in Unity ML-Agents, OpenAI Gym, and many game engines

### 10.2 Robotics and Control
- **Locomotion:** Training robots to walk, run, and navigate complex terrain
- **Manipulation:** Learning dexterous hand movements for object manipulation
- **Sim-to-real transfer:** PPO's stability makes it easier to transfer policies from simulation to real robots

### 10.3 Natural Language Processing and RLHF
- **ChatGPT / GPT-4:** PPO is the core algorithm behind Reinforcement Learning from Human Feedback (RLHF)
- **Alignment:** PPO fine-tunes language models to follow instructions and align with human preferences
- **Why PPO for RLHF?** Its stability is crucial when the "reward" is a learned reward model (which can be noisy)

### 10.4 Autonomous Systems
- **Autonomous driving:** Training driving policies in simulators
- **Drone control:** Navigation and obstacle avoidance
- **Resource management:** Data center cooling, traffic light control

### 10.5 Why PPO is the "Default Choice"
1. **Simplicity:** First-order optimization (no conjugate gradients like TRPO)
2. **Stability:** Clipping prevents catastrophic policy updates
3. **Generality:** Works well for both discrete and continuous actions
4. **Robustness:** Less sensitive to hyperparameters than other policy gradient methods
5. **Scalability:** Easy to parallelize (multiple workers collecting data)
6. **Proven track record:** Successfully deployed in billion-dollar systems

### 10.6 Limitations of PPO
- **Sample inefficiency:** Being on-policy, it cannot reuse old data like off-policy methods
- **Hyperparameter tuning:** While robust, optimal performance still requires tuning
- **Local optima:** Like all policy gradient methods, can get stuck in local optima
- **Reward shaping:** Often requires careful reward engineering for complex tasks


In [ ]:
# 🌐 Step 13: PPO in RLHF — Conceptual Diagram
# Let's create a visual representation of how PPO is used in RLHF

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# Title
ax.text(5, 5.5, 'PPO in RLHF: Aligning Language Models', fontsize=16, fontweight='bold', 
        ha='center', va='center')

# Boxes
boxes = [
    (1, 3.5, 'Pretrained\nLanguage Model', 'lightblue'),
    (4, 3.5, 'Supervised\nFine-Tuning', 'lightgreen'),
    (7, 3.5, 'RLHF with PPO', 'lightcoral'),
    (1, 1.5, 'Human\nPreferences', 'lightyellow'),
    (4, 1.5, 'Reward Model\n(Trained)', 'lightyellow'),
    (7, 1.5, 'Aligned\nPolicy (PPO)', 'lightyellow'),
]

for x, y, text, color in boxes:
    rect = plt.Rectangle((x-0.8, y-0.5), 1.6, 1.0, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, fontweight='bold')

# Arrows
arrows = [
    ((1.8, 3.5), (3.2, 3.5)),
    ((4.8, 3.5), (6.2, 3.5)),
    ((1, 2.5), (1, 2.0)),
    ((1.8, 1.5), (3.2, 1.5)),
    ((4.8, 1.5), (6.2, 1.5)),
    ((7, 2.5), (7, 3.0)),
    ((4, 2.5), (4, 3.0)),
]

for start, end in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

# Labels
ax.text(2.5, 4.0, 'SFT Data', ha='center', fontsize=9, style='italic')
ax.text(5.5, 4.0, 'PPO Update', ha='center', fontsize=9, style='italic')
ax.text(2.5, 1.0, 'Comparison Data', ha='center', fontsize=9, style='italic')
ax.text(5.5, 1.0, 'Reward Signal', ha='center', fontsize=9, style='italic')

# Side note
ax.text(5, 0.3, 'PPO provides stable policy updates crucial when rewards come from a learned (imperfect) reward model',
        ha='center', fontsize=10, style='italic', color='darkred',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()


## 🛠️ Hands-On Exercises

Now it's your turn to solidify your understanding of PPO through hands-on practice! These exercises progress from basic implementation to advanced analysis.


### Exercise 1: Implement the Clipped Surrogate Objective from Scratch
Implement the PPO clipped surrogate loss function manually using only NumPy. Test it with sample ratios and advantages to verify the clipping behavior.


### Exercise 2: Train PPO on CartPole with Custom Hyperparameters
Modify the training configuration (learning rate, clip epsilon, entropy coefficient) and observe how each affects training speed and stability. Document your findings.


### Exercise 3: Experiment with Different Clip Ranges
Train PPO with clip epsilon values of [0.05, 0.1, 0.2, 0.3, 0.5]. Plot the learning curves and analyze which value provides the best stability-sample-efficiency tradeoff.


### Exercise 4: Implement Generalized Advantage Estimation (GAE) Manually
Write a standalone GAE function that takes rewards, values, and dones as input and returns advantages and returns. Verify it against the implementation provided in this notebook.


### Exercise 5: Visualize Training Stability
Run 5 independent training runs with different random seeds. Plot all reward curves together with mean and standard deviation shading to demonstrate PPO's training stability.


### Exercise 6: Compare PPO with a Simple Policy Gradient (REINFORCE)
Implement a basic REINFORCE agent and train it on CartPole. Compare its learning curve, sample efficiency, and final performance with PPO on the same number of timesteps.


### Exercise 7: Tune PPO for Better Sample Efficiency
Experiment with different combinations of batch size, update epochs, and timesteps per update. Try to maximize the average reward per timestep (sample efficiency metric).


### Exercise 8: Analyze the Effect of Entropy Regularization
Train PPO with entropy coefficients of [0.0, 0.001, 0.01, 0.1]. Plot entropy over time and final policy performance. Discuss the exploration-exploitation tradeoff.


### Exercise 9: Train PPO on LunarLander-v2
Adapt the PPO implementation for the LunarLander-v2 environment (continuous action space requires modifications). Compare convergence speed and final performance with CartPole.


### Exercise 10: Discuss Advantages of PPO Over Earlier Methods
Write a detailed analysis (2-3 paragraphs) comparing PPO with TRPO, vanilla Actor-Critic, and DQN. Focus on computational complexity, implementation difficulty, and practical performance.


## Solutions (Review After Attempting)

Below are detailed solutions and explanations for each exercise. **Try the exercises first before looking at these!** The best learning comes from struggling with the problem yourself. 💪


### Solution 1: Clipped Surrogate Objective

The key insight is the `min` operation combined with `clip`. For positive advantages, we cap the ratio at `1 + ε` to prevent over-optimizing good actions. For negative advantages, we floor the ratio at `1 - ε` to prevent over-penalizing bad actions. The `min` ensures we always take the pessimistic (lower) estimate.


In [ ]:
import numpy as np

def ppo_clipped_objective(ratios, advantages, epsilon=0.2):
    """
    Compute PPO clipped surrogate objective.
    
    Args:
        ratios: np.array of probability ratios r_t(θ)
        advantages: np.array of advantage estimates A_t
        epsilon: clipping parameter
    
    Returns:
        objective: scalar objective value (to be maximized)
    """
    # Clip the ratios
    clipped_ratios = np.clip(ratios, 1 - epsilon, 1 + epsilon)
    
    # Unclipped and clipped objectives
    unclipped = ratios * advantages
    clipped = clipped_ratios * advantages
    
    # Take the minimum (pessimistic bound)
    objective = np.minimum(unclipped, clipped)
    
    # Return mean (we maximize this, so gradient ascent)
    return np.mean(objective)

# Test
test_ratios = np.array([0.8, 0.95, 1.0, 1.1, 1.3])
test_advantages = np.array([2.0, 2.0, 2.0, 2.0, 2.0])
print("Ratios:", test_ratios)
print("Advantages:", test_advantages)
print("PPO Objective:", ppo_clipped_objective(test_ratios, test_advantages, epsilon=0.2))
print("\nNote: Ratio 1.3 is clipped to 1.2, reducing its contribution.")


### Solution 2: Custom Hyperparameters

Key observations you should have made:
- **Higher learning rate (e.g., 1e-3):** Faster initial learning but risk of instability and divergence
- **Lower learning rate (e.g., 1e-4):** Slower but more stable; may need more timesteps
- **Higher entropy (e.g., 0.05):** More exploration, slower convergence but better final policy robustness
- **Lower entropy (e.g., 0.001):** Faster convergence but risk of premature convergence to local optima
- **Higher clip (e.g., 0.3):** More aggressive updates, faster learning but less stable
- **Lower clip (e.g., 0.1):** Very conservative, very stable but slower learning


In [ ]:
# Example configuration for faster learning (more aggressive)
FAST_CONFIG = {
    'lr': 5e-4,
    'clip_epsilon': 0.3,
    'entropy_coef': 0.005,
    'epochs': 6,
    'timesteps_per_update': 1024
}

# Example configuration for maximum stability
STABLE_CONFIG = {
    'lr': 1e-4,
    'clip_epsilon': 0.1,
    'entropy_coef': 0.02,
    'epochs': 3,
    'timesteps_per_update': 4096
}

print("Fast config prioritizes speed; Stable config prioritizes reliability.")
print("The default config (lr=3e-4, clip=0.2) is a balanced middle ground.")


### Solution 3: Clip Range Analysis

Expected results:
- **ε = 0.05:** Very stable but slow. The policy barely changes each update. May plateau below optimal.
- **ε = 0.1:** Stable and reasonably fast. Good for sensitive environments.
- **ε = 0.2:** The sweet spot for most tasks. Good balance of speed and stability.
- **ε = 0.3:** Faster learning but occasional instability spikes. May overshoot optimal policies.
- **ε = 0.5:** Often unstable. Large updates can destroy policy performance suddenly.


In [ ]:
# The hyperparameter comparison code in Section 8 demonstrates this.
# Typical finding: ε=0.2 provides the best reward vs stability tradeoff.
print("Key insight: The clip parameter acts as a soft trust region.")
print("Too small = stuck in local optima. Too large = unstable updates.")


### Solution 4: Manual GAE Implementation

GAE works by recursively accumulating TD errors weighted by (γλ)^l. The key is iterating backwards through the trajectory and handling episode boundaries (where the bootstrap value is 0).


In [ ]:
def manual_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Manual GAE implementation with detailed comments."""
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    
    # GAE accumulator
    gae = 0.0
    
    # Iterate backwards through the trajectory
    for t in reversed(range(T)):
        # If episode ended at this step, next value is 0 and we reset GAE
        if dones[t]:
            next_value = 0.0
            gae = 0.0
        else:
            next_value = values[t + 1]
        
        # TD error: δ_t = r_t + γV(s_{t+1}) - V(s_t)
        delta = rewards[t] + gamma * next_value - values[t]
        
        # GAE: Â_t = δ_t + (γλ)δ_{t+1} + (γλ)^2 δ_{t+2} + ...
        gae = delta + gamma * lam * gae
        advantages[t] = gae
    
    returns = advantages + values[:-1]
    return advantages, returns

# Verify against the notebook's implementation
rewards = np.array([1.0, 0.0, 1.0, 0.0, 1.0])
values = np.array([0.5, 0.6, 0.55, 0.7, 0.65, 0.0])
dones = np.array([0, 0, 0, 0, 1])

adv1, ret1 = compute_gae(rewards, values, dones)  # Notebook version
adv2, ret2 = manual_gae(rewards, values, dones)   # Your version

print("Notebook GAE advantages:", np.round(adv1, 4))
print("Manual GAE advantages: ", np.round(adv2, 4))
print("Match:", np.allclose(adv1, adv2))


### Solution 5: Training Stability Visualization

PPO should show relatively low variance across seeds compared to REINFORCE or vanilla Actor-Critic. The clipping mechanism ensures that even with different random initializations, the policy updates stay within a reasonable range.


In [ ]:
# Pseudocode for multi-seed training:
"""
all_rewards = []
for seed in range(5):
    agent = PPOAgent(...)
    rewards = train(agent, seed=seed)
    all_rewards.append(rewards)

# Plot mean and std
max_len = max(len(r) for r in all_rewards)
padded = np.array([np.pad(r, (0, max_len - len(r)), mode='edge') for r in all_rewards])
mean = padded.mean(axis=0)
std = padded.std(axis=0)

plt.plot(mean, label='Mean Reward')
plt.fill_between(range(len(mean)), mean - std, mean + std, alpha=0.3)
"""
print("PPO's stability comes from the clipped objective limiting per-update policy changes.")


### Solution 6: REINFORCE vs PPO Comparison

REINFORCE will likely:
- Learn much slower (updates only per episode)
- Have much higher variance in gradient estimates
- Require a much smaller learning rate to avoid divergence
- Achieve lower final performance on the same timestep budget
- Show more erratic learning curves

PPO's advantages: mini-batch updates, advantage estimation, clipping, and multiple epochs per batch.


In [ ]:
# Simple REINFORCE implementation for comparison
class REINFORCE:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99):
        self.policy = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(),
            nn.Linear(128, action_dim)
        ).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.gamma = gamma
        self.log_probs = []
        self.rewards = []
    
    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        logits = self.policy(state)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        self.log_probs.append(dist.log_prob(action))
        return action.item()
    
    def update(self):
        returns = []
        G = 0
        for r in reversed(self.rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        loss = 0
        for log_prob, G in zip(self.log_probs, returns):
            loss -= log_prob * G
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.log_probs = []
        self.rewards = []

print("REINFORCE is simpler but suffers from high variance and poor sample efficiency.")


### Solution 7: Sample Efficiency Tuning

To improve sample efficiency in PPO:
1. **Increase epochs per update** (e.g., 10 instead of 4) — more gradient steps per sample
2. **Decrease timesteps per update** (e.g., 1024 instead of 2048) — more frequent updates
3. **Use smaller batch sizes** — more noise in updates can help escape local optima
4. **Tune GAE lambda** — lower λ reduces variance but increases bias
5. **Learning rate scheduling** — decay LR as performance plateaus
6. **Normalize observations** — crucial for continuous control tasks

Tradeoff: More epochs increase computational cost per sample but may reduce total samples needed.


In [ ]:
# High sample-efficiency configuration
EFFICIENT_CONFIG = {
    'timesteps_per_update': 1024,   # Update more frequently
    'epochs': 10,                    # Reuse samples more
    'batch_size': 32,               # Smaller batches for noise
    'lr': 2.5e-4,
    'gae_lambda': 0.9,              # Lower variance (more bias)
    'clip_epsilon': 0.1             # Conservative updates
}

print("Sample efficiency = mean reward achieved per environment timestep.")
print("This config trades some stability for more updates per sample.")


### Solution 8: Entropy Regularization Analysis

Expected observations:
- **coef = 0.0:** Fastest initial convergence, but policy may collapse to deterministic suboptimal actions. Entropy will drop to near zero quickly.
- **coef = 0.001:** Mild exploration. Good balance for simple environments.
- **coef = 0.01:** Standard value. Maintains reasonable exploration throughout training.
- **coef = 0.1:** Strong exploration. Entropy stays high, but learning is slower as the policy is forced to stay stochastic.

The entropy bonus is crucial for escaping local optima, especially in environments with deceptive rewards.


In [ ]:
# Entropy decay schedule example
def entropy_schedule(initial_coef, final_coef, total_steps, current_step):
    """Linear decay of entropy coefficient."""
    fraction = min(1.0, current_step / total_steps)
    return initial_coef + (final_coef - initial_coef) * fraction

# Many advanced implementations use entropy decay:
# Start with high exploration (coef=0.01), decay to near zero (coef=0.0001)
print("Entropy decay: encourages exploration early, exploitation later.")


### Solution 9: LunarLander-v2 Adaptation

LunarLander-v2 requires modifications because:
1. **Longer episodes:** May need more timesteps per update (4096+)
2. **Sparse rewards:** GAE with high λ (0.99) helps credit assignment
3. **Observation normalization:** Critical for stable training
4. **Network size:** May need larger hidden layers (512 units)
5. **Reward scaling:** LunarLander rewards are in [-100, 100], may need normalization

For continuous action spaces (LunarLander has discrete actions, but many robotics tasks don't), you would use a Gaussian policy instead of Categorical.


In [ ]:
# LunarLander-specific configuration
LUNAR_CONFIG = {
    'env': 'LunarLander-v2',
    'hidden_dim': 512,
    'lr': 3e-4,
    'gamma': 0.999,           # Higher discount for longer episodes
    'gae_lambda': 0.99,       # Higher lambda for long-term credit
    'clip_epsilon': 0.2,
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    'epochs': 10,
    'batch_size': 64,
    'timesteps_per_update': 4096,
    'total_timesteps': 2_000_000
}

print("LunarLander requires more timesteps and larger networks than CartPole.")
print("Observation normalization is highly recommended for this environment.")


### Solution 10: PPO vs Other Methods — Written Analysis

**PPO vs TRPO:**
TRPO was the first major algorithm to use trust regions for stable policy updates. It enforces a hard KL-divergence constraint using second-order methods (Fisher information matrix and conjugate gradients). While theoretically elegant, TRPO is computationally expensive and complex to implement. PPO achieves nearly identical stability with first-order gradient descent and a simple clipping operation. In practice, PPO matches or exceeds TRPO performance while being orders of magnitude faster and easier to code. This is why PPO largely replaced TRPO in research and industry.

**PPO vs Vanilla Actor-Critic:**
Vanilla Actor-Critic (A2C/A3C) updates the policy using the advantage-weighted log probability. While better than REINFORCE due to bootstrapping, it lacks any mechanism to prevent overly large policy updates. A single bad gradient step can collapse performance, making it extremely sensitive to learning rates. PPO adds the clipped surrogate objective, which acts as a soft trust region. It also typically uses multiple epochs per batch and GAE for better advantage estimation. The result is dramatically improved stability and sample efficiency.

**PPO vs DQN:**
DQN is off-policy and uses experience replay, giving it excellent sample efficiency. However, DQN only works for discrete actions and can suffer from overestimation bias (addressed by Double DQN). PPO is on-policy (less sample efficient) but handles both discrete and continuous actions natively. PPO's policy gradient approach directly optimizes the policy rather than approximating it via Q-values, which often leads to smoother learning in continuous control. For tasks requiring continuous actions (robotics) or where stability is paramount (RLHF), PPO is the clear choice. For discrete tasks with limited environment interaction, DQN may be preferable.


---

## 🎉 Congratulations!

You have now reached modern reinforcement learning. **PPO is one of the most important algorithms used in cutting-edge AI today.** From training game agents to aligning large language models, PPO's combination of simplicity, stability, and performance makes it the workhorse of the RL world.

### Key Takeaways from Today:
- ✅ PPO solves the instability problem of vanilla policy gradients through **clipped surrogate objectives**
- ✅ The probability ratio $r_t(\theta)$ and clipping threshold $\epsilon$ form a soft trust region
- ✅ GAE provides a flexible bias-variance tradeoff for advantage estimation
- ✅ PPO is the default choice for continuous control, robotics, and RLHF
- ✅ Hyperparameter tuning (especially clip epsilon and entropy) is crucial for optimal performance

### Tomorrow's Teaser 🚀
**Day 76: Multi-Agent Reinforcement Learning and Advanced Topics**
Tomorrow we expand beyond single agents and explore how multiple agents learn to cooperate, compete, and communicate in shared environments. We'll cover Multi-Agent PPO (MAPPO), competitive game theory, and emergent behaviors!

---

**Python & AI Learning Path | Day 75 / 369** ⭐

*Keep learning, keep building, and keep pushing the boundaries of what's possible with AI!* 🧠📈
